<a href="https://www.kaggle.com/code/felmirra/narm-whith-anomal?scriptVersionId=311022657" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import gc
import glob 
import numpy as np
import pandas as pd
import datetime as dt
import os

INPUT_DIR = "/kaggle/input/datasets/retailrocket/ecommerce-dataset"
WORK_DIR = "/kaggle/working/retailrocket_project"
STAGE01_DIR = os.path.join(WORK_DIR, "stage_01_retailrocket_preprocessing")

SOURCE_EVENTS = os.path.join(INPUT_DIR, "events.csv")
OUTPUT_DIR = STAGE01_DIR

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("INPUT_DIR   =", INPUT_DIR)
print("WORK_DIR    =", WORK_DIR)
print("STAGE01_DIR =", STAGE01_DIR)
print("SOURCE_EVENTS exists:", os.path.exists(SOURCE_EVENTS))

def build_retailrocket_gru4rec_tables(events_csv_path: str, output_dir: str):
    print("Чтение events.csv ...")
    raw_events = pd.read_csv(events_csv_path)
    print("Исходный размер:", raw_events.shape)
    print("Столбцы:", list(raw_events.columns))

    # Базовая подготовка
    log_df = raw_events.copy()
    log_df["user_id_tmp"] = log_df["visitorid"].values
    log_df["product_id_tmp"] = log_df["itemid"].values

    # Сортировка событий
    log_df = log_df.sort_values(
        ["user_id_tmp", "timestamp", "product_id_tmp"]
    ).reset_index(drop=True)

    # Формирование session_id:
    # новая сессия, если сменился пользователь
    # или если разрыв между соседними событиями > 1 часа
    next_session_flag = np.hstack([
        False,
        (
            (log_df["timestamp"].values[1:] - log_df["timestamp"].values[:-1]) > 3600 * 1000
        ) | (
            log_df["user_id_tmp"].values[1:] != log_df["user_id_tmp"].values[:-1]
        )
    ])
    log_df["session_id_tmp"] = np.cumsum(next_session_flag)

    # Оставляем только просмотры
    views_only = log_df.loc[
        log_df["event"] == "view",
        ["session_id_tmp", "product_id_tmp", "timestamp"]
    ].copy()
    views_only.columns = ["session_id", "item_id", "time"]

    print("\nПосле фильтрации event == 'view':")
    print("Размер:", views_only.shape)
    print("Сессий:", views_only["session_id"].nunique())
    print("Товаров:", views_only["item_id"].nunique())

    # Убираем подряд идущие дубли item внутри одной сессии
    consecutive_duplicate_mask = np.hstack([
        False,
        (
            (views_only["session_id"].values[1:] == views_only["session_id"].values[:-1]) &
            (views_only["item_id"].values[1:] == views_only["item_id"].values[:-1])
        )
    ])

    cleaned_views = views_only.loc[~consecutive_duplicate_mask].copy()

    print("\nПосле удаления подряд идущих дублей:")
    print("Размер:", cleaned_views.shape)
    print("Сессий:", cleaned_views["session_id"].nunique())
    print("Товаров:", cleaned_views["item_id"].nunique())

    # Итеративная очистка:
    # 1) удаляем сессии длины 1
    # 2) удаляем товары с поддержкой < 5
    previous_size = -1
    iteration_num = 0

    while len(cleaned_views) != previous_size:
        iteration_num += 1
        previous_size = len(cleaned_views)

        session_sizes = cleaned_views.groupby("session_id").size()
        valid_sessions = session_sizes[session_sizes > 1].index
        cleaned_views = cleaned_views[cleaned_views["session_id"].isin(valid_sessions)].copy()
        gc.collect()

        item_support = cleaned_views.groupby("item_id").size()
        frequent_items = item_support[item_support >= 5].index
        cleaned_views = cleaned_views[cleaned_views["item_id"].isin(frequent_items)].copy()
        gc.collect()

        print(f"\nИтерация очистки {iteration_num}:")
        print("Размер:", cleaned_views.shape)
        print("Сессий:", cleaned_views["session_id"].nunique())
        print("Товаров:", cleaned_views["item_id"].nunique())

    # Финальный полный датасет
    final_full = pd.DataFrame({
        "SessionId": cleaned_views["session_id"].values,
        "ItemId": cleaned_views["item_id"].values,
        "Time": cleaned_views["time"].values,
    })

    full_path = os.path.join(output_dir, "retailrocket_processed_view_full.tsv")
    final_full.to_csv(full_path, sep="\t", index=False)

    print("\nСохранён полный файл:")
    print(full_path)

    # Временной split как в оригинальном preprocessing
    tsplit_ms = 1442545200000
    one_day_ms = 86400 * 1000

    session_start_time = final_full.groupby("SessionId")["Time"].min()

    test_session_ids = session_start_time[session_start_time >= tsplit_ms - one_day_ms * 7].index
    test_df = final_full[final_full["SessionId"].isin(test_session_ids)].copy()
    train_full_df = final_full[final_full["Time"] < tsplit_ms - one_day_ms * 7].copy()

    # train_full: ещё раз гарантируем длину сессии > 1
    train_session_sizes = train_full_df.groupby("SessionId").size()
    valid_train_sessions = train_session_sizes[train_session_sizes > 1].index
    train_full_df = train_full_df[train_full_df["SessionId"].isin(valid_train_sessions)].copy()

    train_full_path = os.path.join(output_dir, "retailrocket_processed_view_train_full.tsv")
    test_path = os.path.join(output_dir, "retailrocket_processed_view_test.tsv")

    train_full_df.to_csv(train_full_path, sep="\t", index=False)
    test_df.to_csv(test_path, sep="\t", index=False)

    print("\nСохранены train/test:")
    print(train_full_path)
    print(test_path)

    # Дополнительный train/valid split внутри train
    train_inner_df = train_full_df[train_full_df["Time"] < tsplit_ms - one_day_ms * 14].copy()
    valid_session_ids = session_start_time[session_start_time >= tsplit_ms - one_day_ms * 14].index
    valid_df = train_full_df[train_full_df["SessionId"].isin(valid_session_ids)].copy()

    inner_train_sizes = train_inner_df.groupby("SessionId").size()
    valid_inner_train_sessions = inner_train_sizes[inner_train_sizes > 1].index
    train_inner_df = train_inner_df[train_inner_df["SessionId"].isin(valid_inner_train_sessions)].copy()

    train_tr_path = os.path.join(output_dir, "retailrocket_processed_view_train_tr.tsv")
    train_valid_path = os.path.join(output_dir, "retailrocket_processed_view_train_valid.tsv")

    train_inner_df.to_csv(train_tr_path, sep="\t", index=False)
    valid_df.to_csv(train_valid_path, sep="\t", index=False)

    print("\nСохранены train_tr/train_valid:")
    print(train_tr_path)
    print(train_valid_path)

    # Статистика по всем итоговым файлам
    summary_names = []
    summary_num_events = []
    summary_num_sessions = []
    summary_num_items = []
    summary_num_days = []
    summary_start = []
    summary_end = []
    summary_min_len = []
    summary_max_len = []
    summary_avg_len = []
    summary_avg_item_views = []
    summary_min_session_sec = []
    summary_max_session_sec = []

    for tsv_path in glob.glob(os.path.join(output_dir, "retailrocket_processed_view*.tsv")):
        part_df = pd.read_csv(tsv_path, sep="\t", dtype={"ItemId": "str"})

        summary_names.append(os.path.basename(tsv_path))
        summary_num_events.append(len(part_df))
        summary_num_sessions.append(part_df["SessionId"].nunique())
        summary_num_items.append(part_df["ItemId"].nunique())
        summary_num_days.append((part_df["Time"].max() - part_df["Time"].min()) / one_day_ms)

        summary_start.append(
            dt.datetime.utcfromtimestamp(part_df["Time"].min() / 1000).strftime("%Y-%m-%d %H:%M:%S.%f")
        )
        summary_end.append(
            dt.datetime.utcfromtimestamp(part_df["Time"].max() / 1000).strftime("%Y-%m-%d %H:%M:%S.%f")
        )

        per_session_len = part_df.groupby("SessionId").size()
        per_item_views = part_df.groupby("ItemId").size()
        per_session_duration = (
            part_df.groupby("SessionId")["Time"].max() - part_df.groupby("SessionId")["Time"].min()
        ) / 1000.0

        summary_min_len.append(per_session_len.min())
        summary_max_len.append(per_session_len.max())
        summary_avg_len.append(per_session_len.mean())
        summary_avg_item_views.append(per_item_views.mean())
        summary_min_session_sec.append(per_session_duration.min())
        summary_max_session_sec.append(per_session_duration.max())

    stats_df = pd.DataFrame({
        "Dataset": summary_names,
        "NumEvents": summary_num_events,
        "NumSessions": summary_num_sessions,
        "NumItems": summary_num_items,
        "NumDays": summary_num_days,
        "StartTime": summary_start,
        "EndTime": summary_end,
        "AvgItemViews": summary_avg_item_views,
        "MinSessionLength": summary_min_len,
        "MaxSessionLength": summary_max_len,
        "AvgSessionLength": summary_avg_len,
        "MinSessionTime_sec": summary_min_session_sec,
        "MaxSessionTime_sec": summary_max_session_sec,
    })

    stats_path = os.path.join(output_dir, "stats.tsv")
    stats_df.to_csv(stats_path, sep="\t", index=False, float_format="%.4f")

    print("\nИтоговая статистика:")
    display(stats_df)

    print("\nСохранён stats.tsv:")
    print(stats_path)

    return {
        "full": full_path,
        "train_full": train_full_path,
        "test": test_path,
        "train_tr": train_tr_path,
        "train_valid": train_valid_path,
        "stats": stats_path,
    }

saved_files = build_retailrocket_gru4rec_tables(
    events_csv_path=SOURCE_EVENTS,
    output_dir=OUTPUT_DIR
)

print("\nГотовые файлы:")
for key, value in saved_files.items():
    print(f"{key}: {value}")

print("\nФайлы в папке stage_01:")
for fname in sorted(os.listdir(STAGE01_DIR)):
    print(fname)

In [ ]:
# АНОМАЛИИ 2
import os
import gc
import json
import pickle
import numpy as np
import pandas as pd

from functools import lru_cache
from sklearn.ensemble import IsolationForest

# =========================================================
# PATHS
# =========================================================
INPUT_DIR = "/kaggle/input/datasets/retailrocket/ecommerce-dataset"
WORK_DIR = "/kaggle/working/retailrocket_project"
STAGE01_DIR = os.path.join(WORK_DIR, "stage_01_retailrocket_preprocessing")
OUT_DIR = os.path.join(WORK_DIR, "stage_02_anomaly_detection_apply_all")
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(STAGE01_DIR, "retailrocket_processed_view_train_tr.tsv")
VALID_PATH = os.path.join(STAGE01_DIR, "retailrocket_processed_view_train_valid.tsv")
TEST_PATH  = os.path.join(STAGE01_DIR, "retailrocket_processed_view_test.tsv")

ITEM_PROP_1 = os.path.join(INPUT_DIR, "item_properties_part1.csv")
ITEM_PROP_2 = os.path.join(INPUT_DIR, "item_properties_part2.csv")
CATEGORY_TREE = os.path.join(INPUT_DIR, "category_tree.csv")

print("TRAIN_PATH   :", TRAIN_PATH, os.path.exists(TRAIN_PATH))
print("VALID_PATH   :", VALID_PATH, os.path.exists(VALID_PATH))
print("TEST_PATH    :", TEST_PATH, os.path.exists(TEST_PATH))
print("ITEM_PROP_1  :", ITEM_PROP_1, os.path.exists(ITEM_PROP_1))
print("ITEM_PROP_2  :", ITEM_PROP_2, os.path.exists(ITEM_PROP_2))
print("CATEGORY_TREE:", CATEGORY_TREE, os.path.exists(CATEGORY_TREE))
print("OUT_DIR      :", OUT_DIR)

# =========================================================
# CONFIG
# =========================================================
CFG = {
    "iforest_n_estimators": 200,
    "iforest_contamination": 0.02,
    "iforest_random_state": 42,

    # strong anomaly = rule AND iforest
    "strong_logic": "and",

    # если хочешь мягче:
    # "rule_min_flags": 3
    # если хочешь как раньше:
    "rule_min_flags": 2,
}

ISO_FEATURES = [
    "session_length",
    "unique_items",
    "repeat_ratio",
    "session_duration_sec",
    "mean_delta_sec",
    "median_delta_sec",
    "min_delta_sec",
    "events_per_minute",
    "unique_categories",
    "mean_category_depth",
    "cat_jump_count",
    "cat_change_ratio",
    "mean_tree_jump",
    "max_tree_jump",
]

# =========================================================
# HELPERS
# =========================================================
def read_split_tsv(path, split_name):
    df = pd.read_csv(path, sep="\t")
    print(f"\n[{split_name}] loaded:", df.shape)
    print(f"[{split_name}] columns:", list(df.columns))

    required_cols = {"SessionId", "ItemId", "Time"}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"{split_name}: expected columns {required_cols}, got {list(df.columns)}")

    df = df[["SessionId", "ItemId", "Time"]].copy()
    df["SessionId"] = pd.to_numeric(df["SessionId"], errors="coerce")
    df["ItemId"] = pd.to_numeric(df["ItemId"], errors="coerce")
    df["Time"] = pd.to_numeric(df["Time"], errors="coerce")

    df = df.dropna().copy()
    df["SessionId"] = df["SessionId"].astype(np.int64)
    df["ItemId"] = df["ItemId"].astype(np.int64)
    df["Time"] = df["Time"].astype(np.int64)

    df = df.sort_values(["SessionId", "Time", "ItemId"]).reset_index(drop=True)

    print(f"[{split_name}] after cleanup:", df.shape)
    print(f"[{split_name}] unique sessions:", df['SessionId'].nunique())
    print(f"[{split_name}] unique items   :", df['ItemId'].nunique())

    return df


def load_item_category_mapping():
    item_prop1 = pd.read_csv(ITEM_PROP_1)
    item_prop2 = pd.read_csv(ITEM_PROP_2)
    item_props = pd.concat([item_prop1, item_prop2], ignore_index=True)

    print("\nitem_properties total:", item_props.shape)

    cat_props = item_props[item_props["property"] == "categoryid"].copy()
    cat_props["itemid"] = pd.to_numeric(cat_props["itemid"], errors="coerce")
    cat_props["timestamp"] = pd.to_numeric(cat_props["timestamp"], errors="coerce")
    cat_props["categoryid"] = pd.to_numeric(cat_props["value"], errors="coerce")

    cat_props = cat_props.dropna(subset=["itemid", "timestamp", "categoryid"]).copy()
    cat_props["itemid"] = cat_props["itemid"].astype(np.int64)
    cat_props["timestamp"] = cat_props["timestamp"].astype(np.int64)
    cat_props["categoryid"] = cat_props["categoryid"].astype(np.int64)

    # последнее известное categoryid для item
    cat_props = cat_props.sort_values(["itemid", "timestamp"])
    item_to_category = (
        cat_props.groupby("itemid", as_index=False)
        .tail(1)[["itemid", "categoryid"]]
        .drop_duplicates("itemid")
    )

    print("items with category mapping:", len(item_to_category))
    return item_to_category


def load_category_tree():
    cat_tree = pd.read_csv(CATEGORY_TREE)
    cat_tree["categoryid"] = pd.to_numeric(cat_tree["categoryid"], errors="coerce")
    cat_tree["parentid"] = pd.to_numeric(cat_tree["parentid"], errors="coerce")
    cat_tree = cat_tree.dropna(subset=["categoryid"]).copy()
    cat_tree["categoryid"] = cat_tree["categoryid"].astype(np.int64)

    parent_map = {}
    for _, row in cat_tree.iterrows():
        cid = int(row["categoryid"])
        pid = row["parentid"]
        parent_map[cid] = None if pd.isna(pid) else int(pid)

    return parent_map


def build_ancestors_and_depths(parent_map):
    ancestors = {}
    depths = {}

    for cid in parent_map.keys():
        path = []
        visited = set()
        cur = cid

        while cur is not None and cur in parent_map and cur not in visited:
            visited.add(cur)
            path.append(cur)
            cur = parent_map[cur]

        ancestors[cid] = path
        depths[cid] = len(path) - 1

    return ancestors, depths


# глобально заполним позже
ancestors_map = None
depth_map = None

@lru_cache(maxsize=None)
def category_tree_distance(c1, c2):
    if c1 == c2:
        return 0.0
    if c1 not in ancestors_map or c2 not in ancestors_map:
        return np.nan

    anc1 = ancestors_map[c1]
    anc2 = ancestors_map[c2]
    set2 = set(anc2)

    lca = None
    for node in anc1:
        if node in set2:
            lca = node
            break

    if lca is None:
        return np.nan

    d1 = depth_map[c1]
    d2 = depth_map[c2]
    dlca = depth_map[lca]
    return float((d1 - dlca) + (d2 - dlca))


def build_session_features(df, item_to_category, split_name="unknown"):
    work = df.copy()

    # attach category
    work = work.merge(
        item_to_category.rename(columns={"itemid": "ItemId", "categoryid": "CategoryId"}),
        how="left",
        on="ItemId"
    )
    work["CategoryDepth"] = work["CategoryId"].map(depth_map)

    print(f"\n[{split_name}] after category merge:")
    print("shape            :", work.shape)
    print("category coverage:", round(work["CategoryId"].notna().mean(), 6))

    # -------------------------
    # base session features
    # -------------------------
    session_length = work.groupby("SessionId").size().rename("session_length")
    unique_items = work.groupby("SessionId")["ItemId"].nunique().rename("unique_items")

    time_min = work.groupby("SessionId")["Time"].min().rename("time_min")
    time_max = work.groupby("SessionId")["Time"].max().rename("time_max")
    session_duration_sec = ((time_max - time_min) / 1000.0).rename("session_duration_sec")

    unique_categories = work.groupby("SessionId")["CategoryId"].nunique().rename("unique_categories")
    mean_category_depth = work.groupby("SessionId")["CategoryDepth"].mean().rename("mean_category_depth")

    sf = pd.concat(
        [
            session_length,
            unique_items,
            session_duration_sec,
            unique_categories,
            mean_category_depth,
        ],
        axis=1
    ).reset_index()

    sf["repeat_ratio"] = 1.0 - (
        sf["unique_items"] / sf["session_length"].clip(lower=1)
    )

    sf["events_per_minute"] = (
        sf["session_length"] /
        (sf["session_duration_sec"] / 60.0).clip(lower=1e-6)
    )

    # -------------------------
    # delta features
    # -------------------------
    work["PrevTime"] = work.groupby("SessionId")["Time"].shift(1)
    work["delta_sec"] = (work["Time"] - work["PrevTime"]) / 1000.0

    delta_aggs = work.groupby("SessionId")["delta_sec"].agg(
        mean_delta_sec="mean",
        median_delta_sec="median",
        min_delta_sec="min"
    ).reset_index()

    sf = sf.merge(delta_aggs, on="SessionId", how="left")

    # -------------------------
    # category transitions
    # -------------------------
    work["PrevCategoryId"] = work.groupby("SessionId")["CategoryId"].shift(1)

    pair_df = work[
        work["PrevCategoryId"].notna() &
        work["CategoryId"].notna()
    ].copy()

    if len(pair_df) > 0:
        pair_df["PrevCategoryId"] = pair_df["PrevCategoryId"].astype(np.int64)
        pair_df["CategoryId"] = pair_df["CategoryId"].astype(np.int64)

        pair_df["cat_changed"] = (pair_df["PrevCategoryId"] != pair_df["CategoryId"]).astype(np.int8)
        pair_df["tree_jump"] = [
            category_tree_distance(int(a), int(b))
            for a, b in zip(pair_df["PrevCategoryId"].values, pair_df["CategoryId"].values)
        ]

        cat_pair_count = pair_df.groupby("SessionId").size().rename("valid_cat_pairs")
        cat_jump_count = pair_df.groupby("SessionId")["cat_changed"].sum().rename("cat_jump_count")
        cat_change_ratio = pair_df.groupby("SessionId")["cat_changed"].mean().rename("cat_change_ratio")
        mean_tree_jump = pair_df.groupby("SessionId")["tree_jump"].mean().rename("mean_tree_jump")
        max_tree_jump = pair_df.groupby("SessionId")["tree_jump"].max().rename("max_tree_jump")

        cat_features = pd.concat(
            [
                cat_pair_count,
                cat_jump_count,
                cat_change_ratio,
                mean_tree_jump,
                max_tree_jump,
            ],
            axis=1
        ).reset_index()

        sf = sf.merge(cat_features, on="SessionId", how="left")
    else:
        sf["valid_cat_pairs"] = np.nan
        sf["cat_jump_count"] = np.nan
        sf["cat_change_ratio"] = np.nan
        sf["mean_tree_jump"] = np.nan
        sf["max_tree_jump"] = np.nan

    print(f"[{split_name}] session_features shape:", sf.shape)
    return sf


def compute_train_fill_medians(sf_train):
    X_train = sf_train[ISO_FEATURES].copy()
    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    medians = X_train.median(numeric_only=True)
    return medians


def apply_fill_rules(sf, train_medians):
    out = sf.copy()

    out = out.replace([np.inf, -np.inf], np.nan)

    # сначала специальные fill, как у тебя было
    fill_with_zero = [
        "unique_categories",
        "valid_cat_pairs",
        "cat_jump_count",
    ]
    for col in fill_with_zero:
        if col in out.columns:
            out[col] = out[col].fillna(0)

    # потом всё остальное train-median
    for col in ISO_FEATURES:
        if col in out.columns:
            fill_value = train_medians[col] if col in train_medians.index else 0.0
            out[col] = out[col].fillna(fill_value)

    return out


def fit_rule_thresholds(sf_train):
    thresholds = {
        "p99_session_length": float(sf_train["session_length"].quantile(0.99)),
        "p99_unique_items": float(sf_train["unique_items"].quantile(0.99)),
        "p99_events_per_minute": float(sf_train["events_per_minute"].quantile(0.99)),
        "p01_median_delta": float(sf_train["median_delta_sec"].quantile(0.01)),
        "p99_unique_categories": float(sf_train["unique_categories"].quantile(0.99)),
        "p99_cat_change_ratio": float(sf_train["cat_change_ratio"].quantile(0.99)),
        "p99_mean_tree_jump": float(sf_train["mean_tree_jump"].quantile(0.99)),
        "p99_max_tree_jump": float(sf_train["max_tree_jump"].quantile(0.99)),
    }
    return thresholds


def apply_rule_detector(sf, thresholds, rule_min_flags=2):
    out = sf.copy()

    out["flag_long_session"] = (out["session_length"] > thresholds["p99_session_length"]).astype(int)
    out["flag_many_items"] = (out["unique_items"] > thresholds["p99_unique_items"]).astype(int)
    out["flag_fast_behavior"] = (out["events_per_minute"] > thresholds["p99_events_per_minute"]).astype(int)
    out["flag_tiny_gaps"] = (out["median_delta_sec"] < thresholds["p01_median_delta"]).astype(int)
    out["flag_many_categories"] = (out["unique_categories"] > thresholds["p99_unique_categories"]).astype(int)
    out["flag_category_chaos"] = (out["cat_change_ratio"] > thresholds["p99_cat_change_ratio"]).astype(int)
    out["flag_big_tree_jumps"] = (out["mean_tree_jump"] > thresholds["p99_mean_tree_jump"]).astype(int)
    out["flag_extreme_tree_jump"] = (out["max_tree_jump"] > thresholds["p99_max_tree_jump"]).astype(int)

    rule_cols = [
        "flag_long_session",
        "flag_many_items",
        "flag_fast_behavior",
        "flag_tiny_gaps",
        "flag_many_categories",
        "flag_category_chaos",
        "flag_big_tree_jumps",
        "flag_extreme_tree_jump",
    ]

    out["rule_score"] = out[rule_cols].sum(axis=1)
    out["rule_anomaly"] = (out["rule_score"] >= rule_min_flags).astype(int)

    return out


def fit_iforest(sf_train_filled):
    X_train = sf_train_filled[ISO_FEATURES].copy()

    iso = IsolationForest(
        n_estimators=CFG["iforest_n_estimators"],
        contamination=CFG["iforest_contamination"],
        random_state=CFG["iforest_random_state"],
        n_jobs=-1,
    )
    iso.fit(X_train)
    return iso


def apply_iforest(sf_filled, iso):
    out = sf_filled.copy()
    X = out[ISO_FEATURES].copy()

    out["iforest_pred"] = iso.predict(X)              # 1 normal, -1 anomaly
    out["iforest_anomaly"] = (out["iforest_pred"] == -1).astype(int)

    # score_samples: чем меньше, тем более аномально
    out["iforest_score_samples"] = iso.score_samples(X)
    out["iforest_decision_function"] = iso.decision_function(X)

    return out


def combine_detectors(sf, strong_logic="and"):
    out = sf.copy()

    if strong_logic == "and":
        out["strong_anomaly"] = (
            (out["rule_anomaly"] == 1) &
            (out["iforest_anomaly"] == 1)
        ).astype(int)
    elif strong_logic == "or":
        out["strong_anomaly"] = (
            (out["rule_anomaly"] == 1) |
            (out["iforest_anomaly"] == 1)
        ).astype(int)
    else:
        raise ValueError("strong_logic must be 'and' or 'or'")

    return out


def save_split_outputs(split_name, raw_df, sf_final):
    split_stub = f"retailrocket_processed_view_{split_name}"

    features_path = os.path.join(OUT_DIR, f"{split_stub}_session_features.csv")
    anomaly_ids_path = os.path.join(OUT_DIR, f"{split_stub}_strong_anomalous_session_ids.csv")
    cleaned_path = os.path.join(OUT_DIR, f"{split_stub}_cleaned.tsv")
    summary_path = os.path.join(OUT_DIR, f"{split_stub}_summary.csv")

    sf_final.to_csv(features_path, index=False)

    sf_final.loc[sf_final["strong_anomaly"] == 1, ["SessionId"]].to_csv(anomaly_ids_path, index=False)

    strong_sessions = set(sf_final.loc[sf_final["strong_anomaly"] == 1, "SessionId"].tolist())
    cleaned_df = raw_df[~raw_df["SessionId"].isin(strong_sessions)][["SessionId", "ItemId", "Time"]].copy()

    cleaned_df.to_csv(cleaned_path, sep="\t", index=False)

    summary_df = pd.DataFrame([{
        "split": split_name,
        "input_rows": len(raw_df),
        "input_sessions": raw_df["SessionId"].nunique(),
        "input_items": raw_df["ItemId"].nunique(),

        "rule_anomalies": int(sf_final["rule_anomaly"].sum()),
        "iforest_anomalies": int(sf_final["iforest_anomaly"].sum()),
        "strong_anomalies": int(sf_final["strong_anomaly"].sum()),

        "cleaned_rows": len(cleaned_df),
        "cleaned_sessions": cleaned_df["SessionId"].nunique(),
        "cleaned_items": cleaned_df["ItemId"].nunique(),
    }])
    summary_df.to_csv(summary_path, index=False)

    print(f"\n[{split_name}] saved:")
    print(features_path)
    print(anomaly_ids_path)
    print(cleaned_path)
    print(summary_path)

    return {
        "features_path": features_path,
        "anomaly_ids_path": anomaly_ids_path,
        "cleaned_path": cleaned_path,
        "summary_path": summary_path,
        "summary_df": summary_df,
    }


def process_one_split(raw_df, split_name, item_to_category, train_thresholds, train_medians, iso_model):
    sf = build_session_features(raw_df, item_to_category, split_name=split_name)
    sf = apply_fill_rules(sf, train_medians)
    sf = apply_rule_detector(sf, train_thresholds, rule_min_flags=CFG["rule_min_flags"])
    sf = apply_iforest(sf, iso_model)
    sf = combine_detectors(sf, strong_logic=CFG["strong_logic"])
    return sf


# =========================================================
# 1. LOAD REFERENCE TABLES
# =========================================================
item_to_category = load_item_category_mapping()

parent_map = load_category_tree()
ancestors_map, depth_map = build_ancestors_and_depths(parent_map)

print("\ncategory tree loaded")
print("num categories in tree:", len(parent_map))

# =========================================================
# 2. LOAD SPLITS
# =========================================================
train_df = read_split_tsv(TRAIN_PATH, "train_tr")
valid_df = read_split_tsv(VALID_PATH, "train_valid")
test_df  = read_split_tsv(TEST_PATH,  "test")

# =========================================================
# 3. FIT ONLY ON TRAIN
# =========================================================
print("\n================ FIT ON TRAIN_TR ONLY ================")

sf_train_raw = build_session_features(train_df, item_to_category, split_name="train_tr_fit")

# train medians
train_medians = compute_train_fill_medians(sf_train_raw)

# fill train using train medians
sf_train_filled = apply_fill_rules(sf_train_raw, train_medians)

# rule thresholds from TRAIN ONLY
train_thresholds = fit_rule_thresholds(sf_train_filled)

print("\nTrain thresholds:")
for k, v in train_thresholds.items():
    print(f"{k}: {v}")

# fit iforest on TRAIN ONLY
iso_model = fit_iforest(sf_train_filled)

# save fitted artifacts
thresholds_path = os.path.join(OUT_DIR, "train_rule_thresholds.json")
medians_path = os.path.join(OUT_DIR, "train_feature_medians.json")
iforest_path = os.path.join(OUT_DIR, "train_iforest.pkl")
config_path = os.path.join(OUT_DIR, "anomaly_config.json")

with open(thresholds_path, "w", encoding="utf-8") as f:
    json.dump(train_thresholds, f, ensure_ascii=False, indent=2)

with open(medians_path, "w", encoding="utf-8") as f:
    json.dump({k: float(v) for k, v in train_medians.to_dict().items()}, f, ensure_ascii=False, indent=2)

with open(iforest_path, "wb") as f:
    pickle.dump(iso_model, f)

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(CFG, f, ensure_ascii=False, indent=2)

print("\nSaved fitted artifacts:")
print(thresholds_path)
print(medians_path)
print(iforest_path)
print(config_path)

# =========================================================
# 4. APPLY SAME RULES TO TRAIN / VALID / TEST
# =========================================================
print("\n================ APPLY TO ALL SPLITS ================")

sf_train_final = process_one_split(
    raw_df=train_df,
    split_name="train_tr",
    item_to_category=item_to_category,
    train_thresholds=train_thresholds,
    train_medians=train_medians,
    iso_model=iso_model,
)

sf_valid_final = process_one_split(
    raw_df=valid_df,
    split_name="train_valid",
    item_to_category=item_to_category,
    train_thresholds=train_thresholds,
    train_medians=train_medians,
    iso_model=iso_model,
)

sf_test_final = process_one_split(
    raw_df=test_df,
    split_name="test",
    item_to_category=item_to_category,
    train_thresholds=train_thresholds,
    train_medians=train_medians,
    iso_model=iso_model,
)

# =========================================================
# 5. SAVE CLEANED SPLITS
# =========================================================
train_out = save_split_outputs("train_tr", train_df, sf_train_final)
valid_out = save_split_outputs("train_valid", valid_df, sf_valid_final)
test_out  = save_split_outputs("test", test_df, sf_test_final)

# =========================================================
# 6. GLOBAL SUMMARY
# =========================================================
global_summary = pd.concat(
    [
        train_out["summary_df"],
        valid_out["summary_df"],
        test_out["summary_df"],
    ],
    ignore_index=True
)

global_summary_path = os.path.join(OUT_DIR, "global_anomaly_summary.csv")
global_summary.to_csv(global_summary_path, index=False)

print("\n================ GLOBAL SUMMARY ================")
print(global_summary)
print("\nSaved:", global_summary_path)

print("\n================ READY FILES FOR RECOMMENDER ================")
print("TRAIN CLEANED:", train_out["cleaned_path"])
print("VALID CLEANED:", valid_out["cleaned_path"])
print("TEST  CLEANED:", test_out["cleaned_path"])

gc.collect()
print("\nDone.")

In [ ]:
# АДАПТЕР
import os
import gc
import json
import pickle
import numpy as np
import pandas as pd

'''
WORK_DIR = "/kaggle/working/retailrocket_project"
STAGE01_DIR = os.path.join(WORK_DIR, "stage_01_retailrocket_preprocessing")
OUT_DIR = os.path.join(WORK_DIR, "stage_03_narm_preprocessing")

os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(STAGE01_DIR, "retailrocket_processed_view_train_tr.tsv")
VALID_PATH = os.path.join(STAGE01_DIR, "retailrocket_processed_view_train_valid.tsv")
TEST_PATH  = os.path.join(STAGE01_DIR, "retailrocket_processed_view_test.tsv")

print("TRAIN_PATH:", TRAIN_PATH, os.path.exists(TRAIN_PATH))
print("VALID_PATH:", VALID_PATH, os.path.exists(VALID_PATH))
print("TEST_PATH :", TEST_PATH, os.path.exists(TEST_PATH))
print("OUT_DIR   :", OUT_DIR)
'''
WORK_DIR = "/kaggle/working/retailrocket_project"
ANOMALY_DIR = os.path.join(WORK_DIR, "stage_02_anomaly_detection_apply_all")
OUT_DIR = os.path.join(WORK_DIR, "stage_03_narm_preprocessing_cleaned")

os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(ANOMALY_DIR, "retailrocket_processed_view_train_tr_cleaned.tsv")
VALID_PATH = os.path.join(ANOMALY_DIR, "retailrocket_processed_view_train_valid_cleaned.tsv")
TEST_PATH  = os.path.join(ANOMALY_DIR, "retailrocket_processed_view_test_cleaned.tsv")


print("TRAIN_PATH:", TRAIN_PATH, os.path.exists(TRAIN_PATH))
print("VALID_PATH:", VALID_PATH, os.path.exists(VALID_PATH))
print("TEST_PATH :", TEST_PATH, os.path.exists(TEST_PATH))
print("OUT_DIR   :", OUT_DIR)

# =========================================================
# 1. HELPERS
# =========================================================
def load_split(path: str, split_name: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t")
    print(f"\n[{split_name}] исходный размер:", df.shape)
    print(f"[{split_name}] столбцы:", list(df.columns))

    expected = {"SessionId", "ItemId", "Time"}
    if not expected.issubset(df.columns):
        raise ValueError(
            f"[{split_name}] ожидались столбцы {expected}, но получены {set(df.columns)}"
        )

    df = df[["SessionId", "ItemId", "Time"]].copy()

    df["SessionId"] = pd.to_numeric(df["SessionId"], errors="coerce")
    df["ItemId"] = pd.to_numeric(df["ItemId"], errors="coerce")
    df["Time"] = pd.to_numeric(df["Time"], errors="coerce")

    df = df.dropna().copy()

    df["SessionId"] = df["SessionId"].astype(np.int64)
    df["ItemId"] = df["ItemId"].astype(np.int64)
    df["Time"] = df["Time"].astype(np.int64)

    df = df.sort_values(["SessionId", "Time", "ItemId"]).reset_index(drop=True)

    print(f"[{split_name}] после очистки:", df.shape)
    print(f"[{split_name}] уникальных сессий:", df['SessionId'].nunique())
    print(f"[{split_name}] уникальных товаров:", df['ItemId'].nunique())

    return df


def sessions_from_dataframe(df: pd.DataFrame, split_name: str):
    """
    Преобразуем таблицу событий в список сессий:
    одна сессия = список ItemId в хронологическом порядке.
    """
    seqs = []
    sess_ids = []

    for session_id, part in df.groupby("SessionId", sort=False):
        items = part["ItemId"].tolist()
        if len(items) >= 2:
            sess_ids.append(int(session_id))
            seqs.append(items)

    print(f"\n[{split_name}] построено последовательностей:", len(seqs))
    if len(seqs) > 0:
        lengths = pd.Series([len(x) for x in seqs])
        print(f"[{split_name}] длина сессий:")
        print(lengths.describe())

    return sess_ids, seqs


def create_item_index(train_sequences):
    """
    Индексация товаров выполняется ТОЛЬКО по train.
    Индексы начинаются с 1.
    0 оставляем как возможный pad_token.
    """
    mapping = {}
    reverse_mapping = {}
    current_idx = 1

    for seq in train_sequences:
        for item in seq:
            if item not in mapping:
                mapping[item] = current_idx
                reverse_mapping[current_idx] = item
                current_idx += 1

    print("\nРазмер словаря товаров (train only):", len(mapping))
    return mapping, reverse_mapping


def apply_indexing(sequences, item_to_idx, split_name: str):
    """
    Для valid/test оставляем только товары, известные по train.
    После этого сессии длины < 2 удаляются.
    """
    converted = []
    removed_sessions = 0

    for seq in sequences:
        indexed_seq = [item_to_idx[x] for x in seq if x in item_to_idx]
        if len(indexed_seq) >= 2:
            converted.append(indexed_seq)
        else:
            removed_sessions += 1

    print(f"\n[{split_name}] после проекции в train-словарь:")
    print(f"[{split_name}] осталось последовательностей: {len(converted)}")
    print(f"[{split_name}] удалено слишком коротких сессий: {removed_sessions}")

    if len(converted) > 0:
        lengths = pd.Series([len(x) for x in converted])
        print(f"[{split_name}] длина последовательностей после фильтрации:")
        print(lengths.describe())

    return converted


def make_prefix_target_examples(indexed_sequences):
    """
    Из последовательности [1,2,3,4] формируем:
      [1]       -> 2
      [1,2]     -> 3
      [1,2,3]   -> 4
    """
    x_data = []
    y_data = []

    for seq in indexed_sequences:
        for pos in range(1, len(seq)):
            prefix = seq[:pos]
            target = seq[pos]
            x_data.append(prefix)
            y_data.append(target)

    return x_data, y_data


def max_sequence_len(sequences):
    if not sequences:
        return 0
    return max(len(seq) for seq in sequences)


def save_pickle(obj, path):
    with open(path, "wb") as f:
        pickle.dump(obj, f)


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def build_summary(name, original_session_count, indexed_sequences, x_data, y_data):
    seq_lengths = [len(s) for s in indexed_sequences] if indexed_sequences else []
    prefix_lengths = [len(s) for s in x_data] if x_data else []

    row = {
        "split": name,
        "sessions_before_indexing": int(original_session_count),
        "sessions_after_indexing": int(len(indexed_sequences)),
        "examples_count": int(len(x_data)),
        "unique_targets": int(len(set(y_data))) if y_data else 0,
        "avg_session_len": float(np.mean(seq_lengths)) if seq_lengths else 0.0,
        "max_session_len": int(np.max(seq_lengths)) if seq_lengths else 0,
        "avg_prefix_len": float(np.mean(prefix_lengths)) if prefix_lengths else 0.0,
        "max_prefix_len": int(np.max(prefix_lengths)) if prefix_lengths else 0,
    }
    return row


# =========================================================
# 2. LOAD BASE SPLITS
# =========================================================
train_df = load_split(TRAIN_PATH, "train")
valid_df = load_split(VALID_PATH, "valid")
test_df  = load_split(TEST_PATH,  "test")

# =========================================================
# 3. BUILD SESSION SEQUENCES
# =========================================================
train_session_ids, train_raw_sequences = sessions_from_dataframe(train_df, "train")
valid_session_ids, valid_raw_sequences = sessions_from_dataframe(valid_df, "valid")
test_session_ids,  test_raw_sequences  = sessions_from_dataframe(test_df,  "test")

# =========================================================
# 4. TRAIN-ONLY ITEM VOCAB
# =========================================================
item_to_idx, idx_to_item = create_item_index(train_raw_sequences)

# =========================================================
# 5. INDEXING
# =========================================================
train_indexed = apply_indexing(train_raw_sequences, item_to_idx, "train")
valid_indexed = apply_indexing(valid_raw_sequences, item_to_idx, "valid")
test_indexed  = apply_indexing(test_raw_sequences,  item_to_idx, "test")

# =========================================================
# 6. PREFIX -> TARGET SAMPLES
# =========================================================
train_x, train_y = make_prefix_target_examples(train_indexed)
valid_x, valid_y = make_prefix_target_examples(valid_indexed)
test_x, test_y   = make_prefix_target_examples(test_indexed)

print("\nКоличество обучающих примеров:")
print("train:", len(train_x))
print("valid:", len(valid_x))
print("test :", len(test_x))

# =========================================================
# 7. SAVE
# =========================================================
# Основной формат: tuple(list_of_prefixes, list_of_targets)
train_tuple = (train_x, train_y)
valid_tuple = (valid_x, valid_y)
test_tuple  = (test_x, test_y)

save_pickle(train_tuple, os.path.join(OUT_DIR, "train.pkl"))
save_pickle(valid_tuple, os.path.join(OUT_DIR, "valid.pkl"))
save_pickle(test_tuple,  os.path.join(OUT_DIR, "test.pkl"))

# Дополнительно сохраняем полные train-sequences
save_pickle(train_indexed, os.path.join(OUT_DIR, "all_train_sequences.pkl"))

# Словари
save_pickle(item_to_idx, os.path.join(OUT_DIR, "item_to_idx.pkl"))
save_pickle(idx_to_item, os.path.join(OUT_DIR, "idx_to_item.pkl"))

# Метаданные
meta = {
    "num_items": len(item_to_idx),
    "train_sessions": len(train_indexed),
    "valid_sessions": len(valid_indexed),
    "test_sessions": len(test_indexed),
    "train_examples": len(train_x),
    "valid_examples": len(valid_x),
    "test_examples": len(test_x),
    "max_train_session_len": max_sequence_len(train_indexed),
    "max_train_prefix_len": max_sequence_len(train_x),
    "pad_token": 0
}
save_json(meta, os.path.join(OUT_DIR, "meta.json"))

# Краткая статистика
summary_rows = []
summary_rows.append(build_summary("train", len(train_raw_sequences), train_indexed, train_x, train_y))
summary_rows.append(build_summary("valid", len(valid_raw_sequences), valid_indexed, valid_x, valid_y))
summary_rows.append(build_summary("test",  len(test_raw_sequences),  test_indexed,  test_x,  test_y))

summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(OUT_DIR, "summary.csv")
summary_df.to_csv(summary_path, index=False)

print("\nСохранённые файлы:")
for fname in sorted(os.listdir(OUT_DIR)):
    print(fname)

print("\nmeta.json:")
print(json.dumps(meta, ensure_ascii=False, indent=2))

print("\nsummary:")
print(summary_df)

In [ ]:
#НАРМ
import os
import gc
import json
import math
import time
import copy
import pickle
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

'''
WORK_DIR = "/kaggle/working/retailrocket_project"
DATA_DIR = os.path.join(WORK_DIR, "stage_03_narm_preprocessing")
OUT_DIR = os.path.join(WORK_DIR, "stage_04_narm_model")

os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "train.pkl")
VALID_PATH = os.path.join(DATA_DIR, "valid.pkl")
TEST_PATH  = os.path.join(DATA_DIR, "test.pkl")
META_PATH  = os.path.join(DATA_DIR, "meta.json")

print("TRAIN_PATH:", TRAIN_PATH, os.path.exists(TRAIN_PATH))
print("VALID_PATH:", VALID_PATH, os.path.exists(VALID_PATH))
print("TEST_PATH :", TEST_PATH, os.path.exists(TEST_PATH))
print("META_PATH :", META_PATH, os.path.exists(META_PATH))
print("OUT_DIR   :", OUT_DIR)
'''

WORK_DIR = "/kaggle/working/retailrocket_project"
DATA_DIR = os.path.join(WORK_DIR, "stage_03_narm_preprocessing_cleaned")
OUT_DIR = os.path.join(WORK_DIR, "stage_04_narm_model_cleaned")

os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, "train.pkl")
VALID_PATH = os.path.join(DATA_DIR, "valid.pkl")
TEST_PATH  = os.path.join(DATA_DIR, "test.pkl")
META_PATH  = os.path.join(DATA_DIR, "meta.json")

print("TRAIN_PATH:", TRAIN_PATH, os.path.exists(TRAIN_PATH))
print("VALID_PATH:", VALID_PATH, os.path.exists(VALID_PATH))
print("TEST_PATH :", TEST_PATH, os.path.exists(TEST_PATH))
print("META_PATH :", META_PATH, os.path.exists(META_PATH))
print("OUT_DIR   :", OUT_DIR)
# =========================================================
# 1. REPRODUCIBILITY
# =========================================================
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

# =========================================================
# 2. CONFIG
# =========================================================
CFG = {
    "embedding_dim": 64,
    "hidden_size": 100,
    "n_layers": 1,

    "batch_size": 256,
    "epochs": 20,
    "learning_rate": 0.001,
    "weight_decay": 1e-5,

    "dropout_emb": 0.25,
    "dropout_ctx": 0.50,

    "max_len": 50,         
    "eval_k": [1,5,10,15,20],
    "grad_clip": 5.0,

    "early_stopping_patience": 4,
    "num_workers": 0,
}

# =========================================================
# 3. LOAD PREPARED DATA
# =========================================================
def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_x, train_y = load_pickle(TRAIN_PATH)
valid_x, valid_y = load_pickle(VALID_PATH)
test_x, test_y   = load_pickle(TEST_PATH)
meta = load_json(META_PATH)

print("\nLoaded examples:")
print("train:", len(train_x), len(train_y))
print("valid:", len(valid_x), len(valid_y))
print("test :", len(test_x), len(test_y))

# num_items = max index + 1 because 0 is padding
NUM_ITEMS = int(meta["num_items"]) + 1
PAD_TOKEN = int(meta.get("pad_token", 0))

print("NUM_ITEMS (with pad):", NUM_ITEMS)
print("PAD_TOKEN:", PAD_TOKEN)

# =========================================================
# 4. DATASET
# =========================================================
class PrefixTargetDataset(Dataset):
    def __init__(self, prefixes, targets, max_len=None):
        self.prefixes = prefixes
        self.targets = targets
        self.max_len = max_len

        if len(self.prefixes) != len(self.targets):
            raise ValueError("prefixes and targets have different lengths")

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        seq = self.prefixes[idx]
        tgt = self.targets[idx]

        if self.max_len is not None and len(seq) > self.max_len:
            # берём последние элементы префикса
            seq = seq[-self.max_len:]

        return seq, tgt


def narm_collate_fn(batch):
    """
    Возвращает:
      seq_tensor: [T, B]
      lengths: list[int]
      targets: [B]
    """
    sequences, targets = zip(*batch)

    lengths = [len(seq) for seq in sequences]
    order = np.argsort(lengths)[::-1]  # по убыванию длины

    sorted_sequences = [sequences[i] for i in order]
    sorted_targets = [targets[i] for i in order]
    sorted_lengths = [lengths[i] for i in order]

    max_batch_len = max(sorted_lengths)
    batch_size = len(sorted_sequences)

    padded = np.full((max_batch_len, batch_size), PAD_TOKEN, dtype=np.int64)

    for col, seq in enumerate(sorted_sequences):
        padded[:len(seq), col] = seq

    seq_tensor = torch.tensor(padded, dtype=torch.long)
    target_tensor = torch.tensor(sorted_targets, dtype=torch.long)

    return seq_tensor, sorted_lengths, target_tensor


train_ds = PrefixTargetDataset(train_x, train_y, max_len=CFG["max_len"])
valid_ds = PrefixTargetDataset(valid_x, valid_y, max_len=CFG["max_len"])
test_ds  = PrefixTargetDataset(test_x,  test_y,  max_len=CFG["max_len"])

train_loader = DataLoader(
    train_ds,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=CFG["num_workers"],
    collate_fn=narm_collate_fn,
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    collate_fn=narm_collate_fn,
)

test_loader = DataLoader(
    test_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    collate_fn=narm_collate_fn,
)

# =========================================================
# 5. MODEL
# =========================================================
class NARMRetailRocket(nn.Module):
    """
    Собственная реализация NARM в логике оригинального репозитория:
      - embedding
      - GRU over sequence
      - global representation = last hidden state
      - local representation = attentive weighted sum of all hidden states
      - final context = concat(local, global)
      - scoring against all item embeddings
    """
    def __init__(
        self,
        num_items: int,
        embedding_dim: int = 64,
        hidden_size: int = 100,
        n_layers: int = 1,
        dropout_emb: float = 0.25,
        dropout_ctx: float = 0.50,
        pad_idx: int = 0,
    ):
        super().__init__()

        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.pad_idx = pad_idx

        self.item_embedding = nn.Embedding(
            num_embeddings=num_items,
            embedding_dim=embedding_dim,
            padding_idx=pad_idx,
        )

        self.embedding_dropout = nn.Dropout(dropout_emb)

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=n_layers,
            bias=True,
            batch_first=False,
        )

        # attention block
        self.attn_local = nn.Linear(hidden_size, hidden_size, bias=False)
        self.attn_global = nn.Linear(hidden_size, hidden_size, bias=False)
        self.attn_score = nn.Linear(hidden_size, 1, bias=False)

        self.context_dropout = nn.Dropout(dropout_ctx)

        # проекция item embedding в пространство 2*hidden
        self.output_bridge = nn.Linear(embedding_dim, 2 * hidden_size, bias=False)

        self._reset_parameters()

    def _reset_parameters(self):
        nn.init.xavier_uniform_(self.item_embedding.weight)
        if self.pad_idx is not None:
            with torch.no_grad():
                self.item_embedding.weight[self.pad_idx].fill_(0.0)

        for name, param in self.gru.named_parameters():
            if "weight" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)

        nn.init.xavier_uniform_(self.attn_local.weight)
        nn.init.xavier_uniform_(self.attn_global.weight)
        nn.init.xavier_uniform_(self.attn_score.weight)
        nn.init.xavier_uniform_(self.output_bridge.weight)

    def forward(self, seq_tensor, lengths):
        """
        seq_tensor: [T, B]
        lengths: list[int]
        returns logits: [B, num_items]
        """
        embedded = self.item_embedding(seq_tensor)          # [T, B, E]
        embedded = self.embedding_dropout(embedded)

        packed = pack_padded_sequence(
            embedded,
            lengths=lengths,
            batch_first=False,
            enforce_sorted=True,
        )

        packed_out, hidden = self.gru(packed)
        gru_out, _ = pad_packed_sequence(packed_out, batch_first=False)   # [T, B, H]

        # global session representation: last hidden layer last state
        h_global = hidden[-1]                               # [B, H]

        # convert to [B, T, H] for attention
        h_seq = gru_out.permute(1, 0, 2).contiguous()       # [B, T, H]
        seq_mask = (seq_tensor.permute(1, 0) != self.pad_idx).float()  # [B, T]

        # attentive local representation
        q1 = self.attn_local(h_seq)                         # [B, T, H]
        q2 = self.attn_global(h_global).unsqueeze(1)        # [B, 1, H]

        attn_hidden = torch.sigmoid(q1 + q2)                # [B, T, H]
        alpha = self.attn_score(attn_hidden).squeeze(-1)    # [B, T]

        # mask padded positions
        alpha = alpha.masked_fill(seq_mask == 0, float("-inf"))
        alpha = torch.softmax(alpha, dim=1)                 # [B, T]

        h_local = torch.sum(alpha.unsqueeze(-1) * h_seq, dim=1)  # [B, H]

        # final session context
        session_context = torch.cat([h_local, h_global], dim=1)   # [B, 2H]
        session_context = self.context_dropout(session_context)

        # score against all items
        item_matrix = self.output_bridge(self.item_embedding.weight)   # [N, 2H]
        logits = torch.matmul(session_context, item_matrix.transpose(0, 1))  # [B, N]

        # padding token should never be recommended
        logits[:, self.pad_idx] = -1e9

        return logits


model = NARMRetailRocket(
    num_items=NUM_ITEMS,
    embedding_dim=CFG["embedding_dim"],
    hidden_size=CFG["hidden_size"],
    n_layers=CFG["n_layers"],
    dropout_emb=CFG["dropout_emb"],
    dropout_ctx=CFG["dropout_ctx"],
    pad_idx=PAD_TOKEN,
).to(DEVICE)

print(model)

# =========================================================
# 6. METRICS
# =========================================================
@torch.no_grad()
def recall_mrr_at_ks(logits, targets, k_list):
    """
    logits: [B, N]
    targets: [B]
    returns:
        recall_dict: {k: value}
        mrr_dict: {k: value}
    """
    max_k = max(k_list)
    _, topk_idx = torch.topk(logits, k=max_k, dim=1)

    recall_dict = {}
    mrr_dict = {}

    for k in k_list:
        current_topk = topk_idx[:, :k]
        expanded_targets = targets.view(-1, 1).expand_as(current_topk)
        hits = (current_topk == expanded_targets)

        recall = hits.any(dim=1).float().mean().item()

        reciprocal_ranks = []
        for row in range(hits.size(0)):
            hit_positions = torch.nonzero(hits[row], as_tuple=False)
            if len(hit_positions) == 0:
                reciprocal_ranks.append(0.0)
            else:
                rank = int(hit_positions[0].item()) + 1
                reciprocal_ranks.append(1.0 / rank)

        mrr = float(np.mean(reciprocal_ranks))

        recall_dict[k] = recall
        mrr_dict[k] = mrr

    return recall_dict, mrr_dict

# =========================================================
# 7. TRAIN / EVAL LOOPS
# =========================================================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CFG["learning_rate"],
    weight_decay=CFG["weight_decay"],
)

def run_train_epoch(model, loader, optimizer, criterion, device, grad_clip=5.0, k_list=None):
    model.train()

    if k_list is None:
        k_list = [20]

    total_loss = 0.0
    total_batches = 0

    total_recall = {k: 0.0 for k in k_list}
    total_mrr = {k: 0.0 for k in k_list}

    for seq_tensor, lengths, targets in loader:
        seq_tensor = seq_tensor.to(device)
        targets = targets.to(device)

        logits = model(seq_tensor, lengths)
        loss = criterion(logits, targets)

        optimizer.zero_grad()
        loss.backward()

        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()

        batch_recall, batch_mrr = recall_mrr_at_ks(logits, targets, k_list=k_list)

        total_loss += loss.item()
        for k in k_list:
            total_recall[k] += batch_recall[k]
            total_mrr[k] += batch_mrr[k]
        total_batches += 1

    out = {
        "loss": total_loss / max(total_batches, 1),
    }
    for k in k_list:
        out[f"recall@{k}"] = total_recall[k] / max(total_batches, 1)
        out[f"mrr@{k}"] = total_mrr[k] / max(total_batches, 1)

    return out


@torch.no_grad()
def run_eval_epoch(model, loader, criterion, device, k_list=None):
    model.eval()

    if k_list is None:
        k_list = [20]

    total_loss = 0.0
    total_batches = 0

    total_recall = {k: 0.0 for k in k_list}
    total_mrr = {k: 0.0 for k in k_list}

    for seq_tensor, lengths, targets in loader:
        seq_tensor = seq_tensor.to(device)
        targets = targets.to(device)

        logits = model(seq_tensor, lengths)
        loss = criterion(logits, targets)

        batch_recall, batch_mrr = recall_mrr_at_ks(logits, targets, k_list=k_list)

        total_loss += loss.item()
        for k in k_list:
            total_recall[k] += batch_recall[k]
            total_mrr[k] += batch_mrr[k]
        total_batches += 1

    out = {
        "loss": total_loss / max(total_batches, 1),
    }
    for k in k_list:
        out[f"recall@{k}"] = total_recall[k] / max(total_batches, 1)
        out[f"mrr@{k}"] = total_mrr[k] / max(total_batches, 1)

    return out

# =========================================================
# 8. TRAINING
# =========================================================
history = []
best_valid_mrr = -1.0
best_state = None
best_epoch = -1
patience_counter = 0
main_k = 20

print("\n===== START TRAINING =====")
for epoch in range(1, CFG["epochs"] + 1):
    t0 = time.time()

    train_stats = run_train_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=DEVICE,
        grad_clip=CFG["grad_clip"],
        k_list=CFG["eval_k"],
    )

    valid_stats = run_eval_epoch(
        model=model,
        loader=valid_loader,
        criterion=criterion,
        device=DEVICE,
        k_list=CFG["eval_k"],
    )

    elapsed = time.time() - t0

    row = {
        "epoch": epoch,
        "train_loss": train_stats["loss"],
        "valid_loss": valid_stats["loss"],
        "epoch_time_sec": elapsed,
    }

    for k in CFG["eval_k"]:
        row[f"train_recall@{k}"] = train_stats[f"recall@{k}"]
        row[f"train_mrr@{k}"] = train_stats[f"mrr@{k}"]
        row[f"valid_recall@{k}"] = valid_stats[f"recall@{k}"]
        row[f"valid_mrr@{k}"] = valid_stats[f"mrr@{k}"]

    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={row['train_loss']:.4f} | "
        f"valid_loss={row['valid_loss']:.4f} | "
        f"time={row['epoch_time_sec']:.1f}s"
    )

    for k in CFG["eval_k"]:
        print(
            f"  train Recall@{k}={row[f'train_recall@{k}']:.4f} | "
            f"train MRR@{k}={row[f'train_mrr@{k}']:.4f} | "
            f"valid Recall@{k}={row[f'valid_recall@{k}']:.4f} | "
            f"valid MRR@{k}={row[f'valid_mrr@{k}']:.4f}"
        )

    current_valid_mrr = valid_stats[f"mrr@{main_k}"]
    if current_valid_mrr > best_valid_mrr:
        best_valid_mrr = current_valid_mrr
        best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch
        patience_counter = 0
        print(f"  -> new best model at epoch {epoch}")
    else:
        patience_counter += 1
        print(f"  -> no improvement ({patience_counter}/{CFG['early_stopping_patience']})")

    if patience_counter >= CFG["early_stopping_patience"]:
        print("Early stopping triggered.")
        break

# =========================================================
# 9. RESTORE BEST MODEL
# =========================================================
if best_state is not None:
    model.load_state_dict(best_state)

# =========================================================
# 10. FINAL TEST
# =========================================================
test_stats = run_eval_epoch(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=DEVICE,
    k_list=CFG["eval_k"],
)

print("\n===== FINAL TEST =====")
print(f"best_epoch : {best_epoch}")
print(f"test_loss  : {test_stats['loss']:.6f}")
for k in CFG["eval_k"]:
    print(f"test_recall@{k} : {test_stats[f'recall@{k}']:.6f}")
    print(f"test_mrr@{k}    : {test_stats[f'mrr@{k}']:.6f}")

# =========================================================
# 11. SAVE ARTIFACTS
# =========================================================
history_df = pd.DataFrame(history)
history_csv_path = os.path.join(OUT_DIR, "narm_history.csv")
history_pkl_path = os.path.join(OUT_DIR, "narm_history.pkl")
model_path = os.path.join(OUT_DIR, "narm_best_model.pt")
summary_path = os.path.join(OUT_DIR, "narm_summary.json")

history_df.to_csv(history_csv_path, index=False)
with open(history_pkl_path, "wb") as f:
    pickle.dump(history, f)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": CFG,
        "num_items": NUM_ITEMS,
        "pad_token": PAD_TOKEN,
        "best_epoch": best_epoch,
        "best_valid_mrr@20": best_valid_mrr,
    },
    model_path,
)

summary = {
    "best_epoch": int(best_epoch),
    "best_valid_mrr@20": float(best_valid_mrr),
    "test_loss": float(test_stats["loss"]),
    "num_items": int(NUM_ITEMS),
    "train_examples": int(len(train_ds)),
    "valid_examples": int(len(valid_ds)),
    "test_examples": int(len(test_ds)),
}

for k in CFG["eval_k"]:
    summary[f"test_recall@{k}"] = float(test_stats[f"recall@{k}"])
    summary[f"test_mrr@{k}"] = float(test_stats[f"mrr@{k}"])
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\nSaved files:")
for fname in sorted(os.listdir(OUT_DIR)):
    print(fname)

print("\nSummary:")
print(json.dumps(summary, ensure_ascii=False, indent=2))